# Zero-shot Classification with Language Models

The goal of this lab are to:
- Familiarize yourself with Large Language Models
    - Understand prompt-based classification with language models
    - Implement classification using next-token logits
- Try to reproduce results described in a scientific paper
- Implement rigorous experiments

You will in this lab aim to produce detailled results following the methodology presented in
- [Making Pre-trained Language Models Better Few-shot Learners](https://aclanthology.org/2021.acl-long.295.pdf) (Gao et al, 2021)
- [Noisy Channel Language Model Prompting for Few-Shot Text Classification](https://aclanthology.org/2022.acl-long.365.pdf) (Min et al, 2022)


You will:
1. Investigate the zero-shot capabilities of relatively small Language Models on classification by **scoring the labels** of the classification tasks, with various *scoring functions*.
2. Implement a *prompting* fine-tuning which does not require changing the prediction head and allows more flexibility on task definition and data use.

### Part I

Assuming a label set of classes $\mathcal{C}$: usually, a sequence of text $x = (w_1, ..., w_l) \in \mathcal{V}^l$ has an associated label $y$, and the classification model learns to predict $$P(y|x)$$

Here, we assume $\mathcal{C}$ to be included in the vocabulary of the model ($\mathcal{C} \cup \mathcal{V})$; and assuming an input context ${x}$, we:
- Create a **prompt** ${x}'$ = $prompt(x)$; the function $prompt$ is task-dependant and given, for example, in Table 1 of the first paper.
- Instead of generating an answer, we will use **the probability for the first token to be generated** $$P(y|x' = prompt(x)), \forall y \in \mathcal{C}$$
- Use the $\text{Argmax}_{y \in \mathcal{C}} P(y|x')$ as prediction,
    - The first paper uses *label words* as classes.
- Compute the appropriate metric for the dataset and compare to a simple baseline (*i.e*, random or majority draw).


#### What to do ?

Your job is to implement experiments for:
- Several of the datasets that are **used in both papers**,
- Using a small version of one of the models used (for example ```GPT-2```)
- Use the direct probability computation used in **both papers**, and implement more elaborate scores from the second paper, as presented in Table 1.
Look at the appropriate metrics and compare with the results given in the papers !

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

import gc
import torch.nn.functional as F

import numpy as np
from tqdm.auto import tqdm

import pandas as pd


/home/infres/boutrouft-25/projects/DATA709B/venv-3090/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "gpt2-large"
#model_name = "gpt2"
# model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/home/infres/boutrouft-25/projects/DATA709B/venv-3090/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

Check GPU memory

In [3]:
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved()  / 1e9, "GB reserved (cached by PyTorch)")
print(torch.cuda.max_memory_allocated() / 1e9, "GB peak allocated")


3.210018816 GB allocated
3.290431488 GB reserved (cached by PyTorch)
3.210018816 GB peak allocated


Clean GPU memory if needed !

In [9]:
del model          
gc.collect()        
torch.cuda.empty_cache()       


In [14]:
text = "The movie was surprisingly touching and well acted."

In [15]:
prompt = f"Review: {text}\nSentiment:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

logits = model(**inputs).logits[0, -1]

pos_id = tokenizer.encode("positive", add_special_tokens=False)[0]
neg_id = tokenizer.encode("negative", add_special_tokens=False)[0]

print("LM-BFF scores:")
print("positive:", logits[pos_id].item())
print("negative:", logits[neg_id].item())

LM-BFF scores:
positive: -0.15855464339256287
negative: -1.8547154664993286


In [16]:
def score(text, label):
    prompt = f"Sentiment: {label}\nReview: {text}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model(**inputs, labels=inputs["input_ids"])
    return -outputs.loss.item()

print("Noisy channel scores:")
print("positive:", score(text, "positive"))
print("negative:", score(text, "negative"))

Noisy channel scores:
positive: -3.968621253967285
negative: -4.047766208648682


## Datasets configurations

We will study three datasets:
- **SST-2**: Stanford Sentiment Treebank, a binary sentiment classification dataset
- **AGNews**: a 4-class news classification dataset
- **Subj**: a binary subjectivity classification dataset

/!\ I wanted to test TREC dataset but I had an issue with it.

Lets define the configuration for each dataset, including the prompt and the label words to use for each class.

In [39]:
DATASETS = {
    "sst2": {
        "load":      lambda: load_dataset("glue", "sst2")["validation"],  
        "text_key":  "sentence", "label_key": "label",
        "labels":    [0, 1],
        "verbalizers": {0: "terrible", 1: "great"},        
        "direct_prefix":  lambda x: f"{x.strip()} It was",
        "null_prefix":    "It was",
        "channel_prefix": lambda w: f"It was {w}.",
    },
    "ag_news": {
        "load":      lambda: load_dataset("ag_news")["test"],
        "text_key":  "text", "label_key": "label",
        "labels":    [0, 1, 2, 3],
        "verbalizers": {0: "World", 1: "Sports", 2: "Business", 3: "Technology"},
        "direct_prefix":  lambda x: f"{x.strip()} Topic:",
        "null_prefix":    "Topic:",
        "channel_prefix": lambda w: f"Topic: {w}.",
    },
    "subj": {
    "load":      lambda: load_dataset("SetFit/subj")["test"],
    "text_key":  "text", "label_key": "label",
    "labels":    [0, 1],
    "verbalizers": {0: "objective", 1: "subjective"},   # check label_text in the dataset to confirm mapping
    "direct_prefix":  lambda x: f"{x.strip()} This is",
    "null_prefix":    "This is",
    "channel_prefix": lambda w: f"This is {w}.",
},
}

## The conditional log-probability

Everything reduces to one quantity: under a causal LM, the log-probability of a *continuation*
given a *prefix*,

$$\log P_{\text{LM}}(\text{continuation} \mid \text{prefix}) = \sum_{t=1}^{T} \log P_{\text{LM}}(\text{token}_t \mid \text{prefix}, \text{token}_{<t})$$

where $T$ is the number of tokens in the continuation and $\text{token}_{<t}$ denotes the continuation tokens preceding position $t$.

In practice, scores are **length-normalized** to avoid bias toward shorter continuations:

$$\text{score}(\text{continuation} \mid \text{prefix}) = \frac{1}{T}\sum_{t=1}^{T} \log P_{\text{LM}}(\text{token}_t \mid \text{prefix}, \text{token}_{<t})$$

In [17]:
@torch.no_grad()
def conditional_logprob(prefix: str, continuation: str, length_normalize: bool = True) -> float:
    prefix_ids = tokenizer(prefix, return_tensors="pt").input_ids.to(device)
    full_ids   = tokenizer(prefix + continuation, return_tensors="pt").input_ids.to(device)

    n_prefix = prefix_ids.shape[1]
    n_cont   = full_ids.shape[1] - n_prefix
    if n_cont <= 0:
        return float("-inf")

    logits      = model(full_ids).logits            
    cont_logits = logits[:, n_prefix - 1:-1, :]      
    cont_ids    = full_ids[:, n_prefix:]             

    logprobs = F.log_softmax(cont_logits, dim=-1)
    token_lp = logprobs.gather(2, cont_ids.unsqueeze(-1)).squeeze(-1)  # (1, n_cont)
    total    = token_lp.sum().item()
    if length_normalize:
        return total / n_cont
    else:
        return total

## The scoring functions

For a task we need a **verbalizer** `v` mapping each label to a word (e.g. positive -> "great")
and a **template** that turns the input into a prompt (e.g. `"{x} It was MASK."`). Mask = The label to predict.

- **Direct**: put the input before the slot, score the label word.  prefix = `"{x} It was"`, continuation = `" great"`.
- **Direct++**: same, but subtract the score of the label word given a content-free `NULL` prefix .
- **Channel**: reverse the conditioning — put the label expression first and score the *input*.
  prefix = `"It was great."`, continuation = `" {x}"`.

  The score functions are implemented as follow:

In [18]:
def score_direct(configuration, x, label):
    word   = configuration["verbalizers"][label]
    prefix = configuration["direct_prefix"](x)
    return conditional_logprob(prefix, f" {word}", length_normalize=True)

def score_direct_pp(configuration, x, label):
    word    = configuration["verbalizers"][label]
    p_input = conditional_logprob(configuration["direct_prefix"](x), f" {word}", length_normalize=True)
    p_null  = conditional_logprob(configuration["null_prefix"],      f" {word}", length_normalize=True)
    return p_input - p_null            

def score_channel(configuration, x, label):
    word   = configuration["verbalizers"][label]
    prefix = configuration["channel_prefix"](word)
    return conditional_logprob(prefix, f" {x.strip()}", length_normalize=True)

SCORERS = {"direct": score_direct, "direct++": score_direct_pp, "channel": score_channel}

def predict(configuration, x, method):
    scores = [SCORERS[method](configuration, x, label) for label in configuration["labels"]]
    return configuration["labels"][int(np.argmax(scores))]

## Evaluation

The paper uses **accuracy**. 

For the baseline we will use  **random** = 1 / number-of-classes.

We subsample the test set `n_examples` for testing (`n_examples=None` for the full set).

In [19]:

def evaluate(dataset_name, method, n_examples=500, seed=0, verbose=True):
    configuration_dataset  = DATASETS[dataset_name]
    data = configuration_dataset["load"]()
    if n_examples and n_examples < len(data):
        data = data.shuffle(seed=seed).select(range(n_examples))

    print(f"Evaluating {dataset_name}/{method} on {len(data)} data samples")

    gold, correct = [], 0
    for ex in tqdm(data, disable=not verbose, desc=f"{dataset_name}/{method}"):
        x, y = ex[configuration_dataset["text_key"]], ex[configuration_dataset["label_key"]]
        gold.append(y)
        if predict(configuration_dataset, x, method) == y:
            correct += 1

    acc      = correct / len(gold)
    random   = 1.0 / len(configuration_dataset["labels"])
    return {"accuracy": acc, "random": random, "n": len(gold)}

#### Let's run a test on SST-2 on small subset of the dataset (500 examples) to check that everything is working.

In [29]:
results = {}
for method in ["direct", "direct++", "channel"]:
    r = evaluate("sst2", method, n_examples=500)
    results[method] = r
    print(f"SST-2  {method:9s}  acc = {r['accuracy']:.3f}   "
          f"(random = {r['random']:.3f}, n = {r['n']})")

Evaluating sst2/direct on 500 data samples


sst2/direct: 100%|██████████████████████████████████████████████████████████████████████████| 500/500 [00:19<00:00, 26.16it/s]


SST-2  direct     acc = 0.538   (random = 0.500, n = 500)
Evaluating sst2/direct++ on 500 data samples


sst2/direct++: 100%|████████████████████████████████████████████████████████████████████████| 500/500 [00:37<00:00, 13.35it/s]


SST-2  direct++   acc = 0.768   (random = 0.500, n = 500)
Evaluating sst2/channel on 500 data samples


sst2/channel: 100%|█████████████████████████████████████████████████████████████████████████| 500/500 [00:18<00:00, 26.42it/s]

SST-2  channel    acc = 0.756   (random = 0.500, n = 500)


A test of classification on the previous example.

In [30]:
text = "The movie was surprisingly touching and well acted."
configuration  = DATASETS["sst2"]
for method in ["direct", "direct++", "channel"]:
    scores = {configuration["verbalizers"][l]: round(SCORERS[method](configuration, text, l), 3) for l in configuration["labels"]}
    pred   = configuration["verbalizers"][predict(configuration, text, method)]
    print(f"{method:9s} {scores}  -> {pred}")

direct    {'terrible': -8.785, 'great': -4.9}  -> great
direct++  {'terrible': -1.246, 'great': 0.74}  -> great
channel   {'terrible': -3.831, 'great': -3.41}  -> great


Let's do a test on the other datasets, on a small subset of the dataset (500 examples).

In [40]:
for name in ["subj", "ag_news"]:
    print(f"\n=== {name} ===")
    for method in ["direct", "direct++", "channel"]:
        r = evaluate(name, method, n_examples=500)
        print(f"  {method:9s}  acc = {r['accuracy']:.3f}   "
              f"random = {r['random']:.3f}")


=== subj ===


Repo card metadata block was not found. Setting CardData to empty.


Evaluating subj/direct on 500 data samples


subj/direct: 100%|██████████████████████████████████████████████████████████████████████████| 500/500 [00:19<00:00, 25.89it/s]


  direct     acc = 0.466   random = 0.500


...

  direct     acc = 0.676   random = 0.250
Evaluating ag_news/direct++ on 500 data samples


ag_news/direct++: 100%|█████████████████████████████████████████████████████████████████████| 500/500 [01:16<00:00,  6.50it/s]


  direct++   acc = 0.648   random = 0.250
Evaluating ag_news/channel on 500 data samples


ag_news/channel: 100%|██████████████████████████████████████████████████████████████████████| 500/500 [00:39<00:00, 12.75it/s]

  channel    acc = 0.626   random = 0.250


## Benchmarks (Single verbalizer):

In [44]:
datasets_to_run = ["sst2", "ag_news", "subj"]
methods = ["direct", "direct++", "channel"]

results = {}
for dataset_name in datasets_to_run:
    results[dataset_name] = {}
    for method in methods:
        r = evaluate(dataset_name, method, n_examples=None)
        results[dataset_name][method] = r
        print(f"{dataset_name:10s} {method:9s}  acc = {r['accuracy']:.3f}   "
              f"(random = {r['random']:.3f}, n = {r['n']})")

Evaluating sst2/direct on 872 data samples


sst2/direct: 100%|██████████████████████████████████████████████████████████████████████████| 872/872 [00:33<00:00, 25.72it/s]


sst2       direct     acc = 0.547   (random = 0.500, n = 872)
Evaluating sst2/direct++ on 872 data samples


sst2/direct++: 100%|████████████████████████████████████████████████████████████████████████| 872/872 [01:08<00:00, 12.73it/s]


sst2       direct++   acc = 0.783   (random = 0.500, n = 872)
Evaluating sst2/channel on 872 data samples


...

subj       direct++   acc = 0.671   (random = 0.500, n = 2000)


Repo card metadata block was not found. Setting CardData to empty.


Evaluating subj/channel on 2000 data samples


subj/channel: 100%|███████████████████████████████████████████████████████████████████████| 2000/2000 [01:16<00:00, 26.07it/s]

subj       channel    acc = 0.571   (random = 0.500, n = 2000)


Display results

In [47]:
table = pd.DataFrame({
    dataset_name: {method: results[dataset_name][method]["accuracy"] * 100
                    for method in methods}
    for dataset_name in datasets_to_run
}).T

table = table[methods]          
table = table.round(1)
table

,direct,direct++,channel
sst2,54.7,78.3,74.4
ag_news,65.9,66.2,59.0
subj,49.2,67.2,57.0


Obtained results with one verbalizer for each dataset and method:
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>direct</th>
      <th>direct++</th>
      <th>channel</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>sst2</th>
      <td>54.7</td>
      <td>78.3</td>
      <td>74.4</td>
    </tr>
    <tr>
      <th>ag_news</th>
      <td>65.9</td>
      <td>66.2</td>
      <td>59.0</td>
    </tr>
    <tr>
      <th>subj</th>
      <td>49.2</td>
      <td>67.2</td>
      <td>57.0</td>
    </tr>
  </tbody>
</table>
</div>

With GPT-2 large and a single verbalizer : Direct++ and channel clearly beats direct on SST-2 .


In order to reproduce what was done in the paper , we have to do four runs (Verbalizers) for each dataset and each method.

We should also report the average accuracy over the four verbalizers, and the worst-case accuracy (the minimum across verbalizers). 

## Benchmarks (4 verbalizers):

In [20]:
DATASET_TEMPLATES = {
    "sst2": {
        "load": lambda: load_dataset("glue", "sst2")["validation"],
        "text_key": "sentence", "label_key": "label",
        "labels": [0, 1],
        "verbalizers": {0: "terrible", 1: "great"},
        "templates": ["A MASK one.", "It was MASK.", "All in all MASK.", "A MASK piece."],
    },
    "ag_news": {
        "load": lambda: load_dataset("ag_news")["test"],
        "text_key": "text", "label_key": "label",
        "labels": [0, 1, 2, 3],
        "verbalizers": {0: "World", 1: "Sports", 2: "Business", 3: "Technology"},
        "templates": ["Topic: MASK.", "Subject: MASK.", "This is about MASK.", "It is about MASK."],
    },
    "subj": {
        "load": lambda: load_dataset("SetFit/subj")["test"],
        "text_key": "text", "label_key": "label",
        "labels": [0, 1],
        "verbalizers": {0: "objective", 1: "subjective"},
        "templates": ["This is MASK.", "It's all MASK.", "It's MASK.", "Is it MASK?"],
    },
}

In [21]:
def make_configuration_from_template(template, base):
    before, _after = template.split("MASK", 1)
    before = before.rstrip()
    return {
        "load": base["load"], "text_key": base["text_key"], "label_key": base["label_key"],
        "labels": base["labels"], "verbalizers": base["verbalizers"],
        "direct_prefix":  lambda x: f"{x.strip()} {before}",
        "null_prefix":    before,
        "channel_prefix": lambda w: template.replace("MASK", w),
    }

In [22]:
def evaluate_configuration(cfg, method, n_examples=500, seed=0, verbose=True):
    data = cfg["load"]()
    if n_examples and n_examples < len(data):
        data = data.shuffle(seed=seed).select(range(n_examples))
    gold, correct = [], 0
    for ex in tqdm(data, disable=not verbose):
        x, y = ex[cfg["text_key"]], ex[cfg["label_key"]]
        gold.append(y)
        correct += int(predict(cfg, x, method) == y)
    return {"accuracy": correct / len(gold), "n": len(gold)}



In [23]:
def zero_shot_4runs(dataset_name, method, n_examples=500, seed=0, verbose=False):
    base = DATASET_TEMPLATES[dataset_name]
    accuracies = []
    for i, template in enumerate(base["templates"]):
        cfg = make_configuration_from_template(template, base)
        r = evaluate_configuration(cfg, method, n_examples=n_examples, seed=seed, verbose=verbose)
        accuracies.append(r["accuracy"])
        print(f"    run {i+1}/4  template={template!r:25s} acc={r['accuracy']:.3f}")
    return {"avg": float(np.mean(accuracies)), "worst": float(np.min(accuracies)), "runs": accuracies}

In [24]:
zero_shot_results = {}
for dataset_name in ["sst2", "ag_news", "subj"]:
    zero_shot_results[dataset_name] = {}
    for method in ["direct", "direct++", "channel"]:
        print(f"\n=== {dataset_name} / {method} (4 runs) ===")
        res = zero_shot_4runs(dataset_name, method, n_examples=None)
        zero_shot_results[dataset_name][method] = res
        print(f"  -> avg={res['avg']*100:.1f}  worst={res['worst']*100:.1f}")


=== sst2 / direct (4 runs) ===


    run 1/4  template='A MASK one.'             acc=0.509
    run 2/4  template='It was MASK.'            acc=0.547
    run 3/4  template='All in all MASK.'        acc=0.509
    run 4/4  template='A MASK piece.'           acc=0.509
  -> avg=51.9  worst=50.9

=== sst2 / direct++ (4 runs) ===
    run 1/4  template='A MASK one.'             acc=0.768
    run 2/4  template='It was MASK.'            acc=0.783
    run 3/4  template='All in all MASK.'        acc=0.813
    run 4/4  template='A MASK piece.'           acc=0.768
  -> avg=78.3  worst=76.8

=== sst2 / channel (4 runs) ===
    run 1/4  template='A MASK one.'             acc=0.735
    run 2/4  template='It was MASK.'            acc=0.744
    run 3/4  template='All in all MASK.'        acc=0.771
    run 4/4  template='A MASK piece.'           acc=0.710
  -> avg=74.0  worst=71.0

=== ag_news / direct (4 runs) ===
    run 1/4  template='Topic: MASK.'            acc=0.659
    run 2/4  template='Subject: MASK.'          acc=0.665
    run 

Repo card metadata block was not found. Setting CardData to empty.


    run 1/4  template='This is MASK.'           acc=0.492


Repo card metadata block was not found. Setting CardData to empty.


...

    run 2/4  template="It's all MASK."          acc=0.536


Repo card metadata block was not found. Setting CardData to empty.


    run 3/4  template="It's MASK."              acc=0.546


Repo card metadata block was not found. Setting CardData to empty.


    run 4/4  template='Is it MASK?'             acc=0.507
  -> avg=54.0  worst=50.7


In [26]:

rows = {
    dataset_name: {
        method: f"{r['avg']*100:.1f}/{r['worst']*100:.1f}"
        for method, r in methods_dict.items()
    }
    for dataset_name, methods_dict in zero_shot_results.items()
}
table = pd.DataFrame(rows).T[["direct", "direct++", "channel"]]
table

,direct,direct++,channel
sst2,51.9/50.9,78.3/76.8,74.0/71.0
ag_news,48.3/27.4,66.5/61.9,60.6/59.0
subj,55.7/48.3,59.1/52.5,54.0/50.7


Obtained results with four verbalizers for each dataset and method:
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>direct</th>
      <th>direct++</th>
      <th>channel</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>sst2</th>
      <td>51.9/50.9</td>
      <td>78.3/76.8</td>
      <td>74.0/71.0</td>
    </tr>
    <tr>
      <th>ag_news</th>
      <td>48.3/27.4</td>
      <td>66.5/61.9</td>
      <td>60.6/59.0</td>
    </tr>
    <tr>
      <th>subj</th>
      <td>55.7/48.3</td>
      <td>59.1/52.5</td>
      <td>54.0/50.7</td>
    </tr>
  </tbody>
</table>
</div>

## Conclusions:

- Direct++ and Channel reproduce the paper's results closely, while plain Direct shows the same instability across verbalizers (As the paper itself reports).

- The channel models shows stability compared to direct and direct++ methods (We see that the channel model is less sensitive to the choice of verbalizer).

